<a href="https://colab.research.google.com/github/narayananxyz/deep_learning/blob/main/capstone/capture_and_predict_from_camera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# Load pretrained model
!pip install gdown -q
import gdown

gdown.download(
    "https://drive.google.com/uc?id=1iZnMCWGq-tW29LqbJmtMalkLz9ifVSMe",
    "cnn_v2_rgb_final.keras",
    quiet=False
)

from tensorflow.keras.models import load_model
model_loaded = load_model("cnn_v2_rgb_final.keras")
print("Model ready!")

Downloading...
From (original): https://drive.google.com/uc?id=1iZnMCWGq-tW29LqbJmtMalkLz9ifVSMe
From (redirected): https://drive.google.com/uc?id=1iZnMCWGq-tW29LqbJmtMalkLz9ifVSMe&confirm=t&uuid=a2b448cc-bf05-45b6-851c-caecafce3de7
To: /content/cnn_v2_rgb_final.keras
100%|██████████| 33.2M/33.2M [00:00<00:00, 149MB/s]


Model ready!


In [ ]:
def capture_and_predict(model):
    class_labels = ['happy', 'neutral', 'sad', 'surprise']

    print("Camera starting... smile! 📸")
    img_data = start_camera()

    img_bytes = b64decode(img_data.split(',')[1])
    img = Image.open(io.BytesIO(img_bytes))
    img = np.array(img)

    # Get image dimensions
    h, w = img.shape[:2]

    # Detect face
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    )
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.05,  # more sensitive
        minNeighbors=3,          # less strict
        minSize=(80, 80)
    )

    if len(faces) == 0:
        print("No face detected — using center crop fallback")
        # Fallback: crop center 60% of frame
        y1, y2 = int(h*0.1), int(h*0.7)
        x1, x2 = int(w*0.2), int(w*0.8)
        face_crop = img[y1:y2, x1:x2]
    else:
        print(f"Face detected!")
        x, y, fw, fh = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)[0]
        # Add padding around face
        pad = 20
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(w, x + fw + pad)
        y2 = min(h, y + fh + pad)
        face_crop = img[y1:y2, x1:x2]

        # Draw box
        img_display = img.copy()
        cv2.rectangle(img_display, (x1, y1), (x2, y2), (0, 255, 0), 3)
        display(Image.fromarray(img_display))

    # Show cropped face
    display(Image.fromarray(face_crop).resize((200, 200)))

    # Preprocess
    face_gray = cv2.cvtColor(face_crop, cv2.COLOR_RGB2GRAY)
    face_resized = cv2.resize(face_gray, (48, 48))
    face_norm = face_resized / 255.0
    face_input = np.expand_dims(face_norm, axis=-1)
    face_input = np.expand_dims(face_input, axis=0)

    # Predict
    predictions = model.predict(face_input, verbose=0)
    predicted_class = class_labels[np.argmax(predictions)]
    confidence = np.max(predictions) * 100

    print(f"\n{'='*40}")
    print(f"  Detected: {predicted_class.upper()} ({confidence:.1f}%)")
    print(f"{'='*40}")
    print("\nAll probabilities:")
    for label, prob in zip(class_labels, predictions[0]):
        bar = '█' * int(prob * 20)
        print(f"  {label:10s} {prob*100:5.1f}% {bar}")

    return predicted_class, confidence

capture_and_predict(model_loaded)